# Notebook 4: SCIGEN — Constrained Crystal Generation (Capstone)

**APS Tutorial T4 — Generative AI for Physics: From Models to Materials**

This is the capstone notebook. Everything from the previous sessions comes together here:
- **Notebook 01:** We learned to represent crystal structures as (L, X, A) and saw that kagome materials are rare
- **Notebook 02:** We trained a DDPM on MNIST and understood how diffusion generates data from noise
- **Notebook 03:** We applied diffusion to crystals (DiffCSP) — wrapped normals, lattice noise, predictor-corrector sampling

Now we will **generate new crystal structures** with targeted geometric constraints
using the SCIGEN diffusion model. SCIGEN adds **inpainting** to the DiffCSP framework:
you specify a sublattice geometry (kagome, honeycomb, triangular, ...) and the model
generates a complete, chemically reasonable crystal around it.

**Paper:** [Structural constraint integration in a generative model for the discovery of quantum materials](https://doi.org/10.1038/s41563-025-02355-y) (Nature Materials, 2025)

> **Prerequisites:** Run `00_setup.ipynb` first (installs dependencies and downloads model weights).

In [ ]:
# takes long (~5-10 min first run) — clones repo, installs deps, downloads model

# --- Run once per Colab session (skip if you already ran 00_setup.ipynb) ---
import os, shutil, subprocess, sys

REPO_URL = 'https://github.com/RyotaroOKabe/APS_demo_SCIGEN.git'
PROJECT_DIR = '/content/APS_demo_SCIGEN'

# Clone repo if needed
if not os.path.exists(os.path.join(PROJECT_DIR, '.git')):
    if os.path.exists(PROJECT_DIR):
        shutil.rmtree(PROJECT_DIR)
    os.system(f'git clone {REPO_URL} {PROJECT_DIR}')

# Install PyG extensions (need matching torch+CUDA wheels)
try:
    import torch_scatter
except ImportError:
    import torch
    _vp = torch.__version__.split('+')[0].split('.')
    tv = f'{_vp[0]}.{_vp[1]}.0'
    if torch.version.cuda:
        cv = 'cu' + torch.version.cuda.replace('.', '')
    else:
        cv = 'cpu'
    whl = f'https://data.pyg.org/whl/torch-{tv}+{cv}.html'
    print(f'Installing PyG extensions from: {whl}')
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q',
                           'torch-scatter', 'torch-sparse', 'torch-cluster',
                           '-f', whl])

# Install all tutorial dependencies from the shared requirements file
subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q',
                       '-r', f'{PROJECT_DIR}/requirements-colab.txt'])

# Download model weights if not present
from pathlib import Path
MODEL_PATH = Path(PROJECT_DIR) / 'models' / 'mp_20'
if not MODEL_PATH.exists() or not list(MODEL_PATH.glob('*.ckpt')):
    import json, zipfile
    from urllib.request import urlopen, urlretrieve
    MODEL_PATH.mkdir(parents=True, exist_ok=True)
    article = json.loads(urlopen('https://api.figshare.com/v2/articles/27778134').read().decode())
    for f in article.get('files', []):
        dest = MODEL_PATH / f['name']
        print(f'Downloading {f["name"]}...')
        urlretrieve(f['download_url'], str(dest))
        if dest.suffix == '.zip':
            with zipfile.ZipFile(dest, 'r') as zf:
                zf.extractall(MODEL_PATH)
            dest.unlink()

os.environ.setdefault('PROJECT_ROOT', PROJECT_DIR)
print('Ready.')

---
## 4.1 What is SCIGEN?

SCIGEN (Structural Constraint Integration in GENerative model) is a diffusion-based
generative model for crystal structure discovery. Its key innovation:

**You specify a geometric constraint** (kagome, honeycomb, triangular, etc.)
**and SCIGEN generates complete crystal structures** that contain that sublattice.

The model uses an **inpainting** approach:
1. The diffusion model generates the full structure (all atoms) from noise
2. The constrained atom positions (defining the desired geometric pattern) are **inpainted** — revealed at the end, replacing the model's predictions at those sites
3. During training, the model learns to anticipate these constraints and generate complementary atoms that form physically reasonable crystals around the target sublattice
4. The lattice geometry is also constrained (e.g., hexagonal cell for kagome)

In the published work, this pipeline generated 10 million candidate structures,
which were screened down to 24,743 DFT-validated materials.

### 4.1.1 SCIGEN pipeline overview

![SCIGEN pipeline](../assets/figure1.png)

### 4.1.2 Available structural constraints

![Lattice types](../assets/SI_arch_lattice_unit_bk.png)

---
## 4.2 Environment setup

If you ran `00_setup.ipynb`, the environment is already configured.
If running this notebook standalone, the cells below handle setup.

In [ ]:
import os, sys

PROJECT_DIR = os.environ.get('PROJECT_ROOT', '/content/APS_demo_SCIGEN')

# Set env vars if not already set (e.g., running standalone)
os.environ.setdefault('PROJECT_ROOT', PROJECT_DIR)
os.environ.setdefault('HYDRA_JOBS', PROJECT_DIR)
os.environ.setdefault('WANDB_DIR', os.path.join(PROJECT_DIR, 'wandb'))
os.makedirs(os.environ['WANDB_DIR'], exist_ok=True)

if PROJECT_DIR not in sys.path:
    sys.path.insert(0, PROJECT_DIR)
if os.path.join(PROJECT_DIR, 'script') not in sys.path:
    sys.path.insert(0, os.path.join(PROJECT_DIR, 'script'))

os.chdir(PROJECT_DIR)
print(f'Working directory: {os.getcwd()}')

---
## 4.3 Load the pretrained model

We load the SCIGEN diffusion model using Hydra for configuration and manual
state-dict loading for compatibility with any PyTorch Lightning version.

In [ ]:
import torch
import numpy as np
import hydra
from hydra import compose, initialize_config_dir
from hydra.core.global_hydra import GlobalHydra
from pathlib import Path

import scigen
print(f'scigen imported from: {scigen.__path__[0]}')


def load_model_for_inference(model_path, device='cpu'):
    """Load SCIGEN pretrained model for inference.

    Uses manual state_dict loading instead of pytorch_lightning's
    load_from_checkpoint, so it works with any PL version.
    """
    model_path = Path(model_path)

    # Clear previous hydra state (allows re-running this cell)
    GlobalHydra.instance().clear()

    # Load model config from hparams.yaml
    with initialize_config_dir(config_dir=str(model_path.resolve()), version_base=None):
        cfg = compose(config_name='hparams')

    # Instantiate model architecture (empty weights)
    model = hydra.utils.instantiate(
        cfg.model, optim=cfg.optim, data=cfg.data,
        logging=cfg.logging, _recursive_=False,
    )

    # Find checkpoint file
    ckpts = sorted(model_path.glob('*.ckpt'))
    if not ckpts:
        raise FileNotFoundError(f'No .ckpt files found in {model_path}')

    ckpt_path = next((c for c in ckpts if 'last' in c.name), ckpts[-1])
    print(f'Loading checkpoint: {ckpt_path.name}')

    checkpoint = torch.load(ckpt_path, map_location='cpu', weights_only=False)
    model.load_state_dict(checkpoint['state_dict'], strict=False)

    # Load data scalers
    for attr, fname in [('lattice_scaler', 'lattice_scaler.pt'),
                        ('scaler', 'prop_scaler.pt')]:
        fpath = model_path / fname
        if fpath.exists():
            setattr(model, attr, torch.load(fpath, map_location='cpu', weights_only=False))

    model = model.to(device)
    model.eval()
    return model, cfg

In [ ]:
MODEL_PATH = Path(PROJECT_DIR) / 'models' / 'mp_20'
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {device}')

model, cfg = load_model_for_inference(MODEL_PATH, device=device)

# Attach the generation sampling method
from scigen.pl_modules.diffusion_w_type import sample_scigen
model.sample_scigen = sample_scigen.__get__(model)

num_params = sum(p.numel() for p in model.parameters())
print(f'Model loaded: {num_params:,} parameters')

---
## 4.4 Choose your structural constraint

This is the key feature of SCIGEN: you pick a lattice geometry, and the model
generates complete crystal structures containing that sublattice.

| Code | Lattice | Known atoms | Description |
|------|---------|-------------|-------------|
| `kag` | **Kagome** | 3 | Corner-sharing triangles — frustrated magnets |
| `hon` | **Honeycomb** | 2 | Graphene-like — topological materials |
| `tri` | **Triangular** | 1 | Simplest 2D lattice |
| `sqr` | **Square** | 1 | Square lattice |
| `lieb` | **Lieb** | 3 | Square with flat bands |
| `elt` | Elongated triangular | 2 | Rectangular + triangular |
| `sns` | Snub square | 4 | Archimedean tiling |
| `tsq` | Truncated square | 4 | Squares + octagons |
| `srt` | Small rhombotrihexagonal | 6 | Complex tiling |
| `snh` | Snub hexagonal | 6 | Complex tiling |
| `trh` | Truncated hexagonal | 6 | Complex tiling |
| `grt` | Great rhombotrihexagonal | 12 | Most complex tiling |
| `van` | Vanilla (no constraint) | 1 | Unconditional generation |

In [ ]:
from torch_geometric.data import DataLoader
from gen_utils import SampleDataset

# ============================================================
#  EDIT THESE PARAMETERS to try different generation settings
# ============================================================
SC_TYPE    = 'kag'    # Structural constraint (see table above)
ELEMENT    = 'Mn'     # Element for constrained sites
BATCH_SIZE = 4        # Structures per batch (keep small for Colab)
NUM_BATCHES = 1       # Number of batches
# ============================================================

FRAC_Z   = 0.5        # z-coordinate of the 2D pattern layer
STEP_LR  = 5e-6       # Diffusion step size
SEED     = 42

# Atom count ranges per structural constraint
SC_NATM_RANGE = {
    'tri': [1, 4],  'hon': [1, 8],  'kag': [1, 12],
    'sqr': [1, 4],  'elt': [1, 8],  'sns': [1, 16],
    'tsq': [1, 16], 'van': [1, 20], 'lieb': [1, 12],
    'srt': [1, 20], 'snh': [1, 20], 'trh': [1, 20],
    'grt': [1, 20],
}

total_structures = BATCH_SIZE * NUM_BATCHES
natm_range = SC_NATM_RANGE.get(SC_TYPE, [1, 20])

print(f'Generation settings:')
print(f'  Lattice type:      {SC_TYPE}')
print(f'  Element:           {ELEMENT}')
print(f'  Structures:        {total_structures}')
print(f'  Atoms per cell:    {natm_range[0]}-{natm_range[1]}')

In [ ]:
# takes long (~20-30 sec) — runs diffusion sampling to generate crystal structures

import time
from tqdm.auto import tqdm

# Build the sample dataset with structural constraints
test_set = SampleDataset(
    dataset='mp_20',
    natm_range=natm_range,
    total_num=total_structures,
    bond_sigma_per_mu=None,
    use_min_bond_len=False,
    known_species=[ELEMENT],
    sc_list=[SC_TYPE],
    frac_z=FRAC_Z,
    c_vec_cons={'scale': None, 'vert': False},
    reduced_mask=False,
    seed=SEED,
    device=device,
)

test_loader = DataLoader(test_set, batch_size=BATCH_SIZE)

# Run diffusion sampling
all_frac_coords = []
all_atom_types = []
all_lattices = []
all_num_atoms = []
all_num_known = []

print(f'Running diffusion (step_lr={STEP_LR})...')
start_time = time.time()

for idx, batch in enumerate(tqdm(test_loader, desc='Generating structures')):
    if torch.cuda.is_available():
        batch.cuda()
    outputs, traj = model.sample_scigen(batch, step_lr=STEP_LR)

    all_frac_coords.append(outputs['frac_coords'].detach().cpu())
    raw_types = outputs['atom_types'].detach().cpu()
    if raw_types.dim() == 2:
        raw_types = raw_types.argmax(dim=-1) + 1
    all_atom_types.append(raw_types)
    all_lattices.append(outputs['lattices'].detach().cpu())
    all_num_atoms.append(outputs['num_atoms'].detach().cpu())
    all_num_known.append(outputs['num_known'].detach().cpu())

    print(f'  Batch {idx + 1}/{len(test_loader)} complete')

elapsed = time.time() - start_time
print(f'\nGeneration complete in {elapsed:.1f}s ({elapsed / total_structures:.1f}s per structure)')

# Concatenate results
frac_coords = torch.cat(all_frac_coords, dim=0)
atom_types = torch.cat(all_atom_types, dim=0)
lattices = torch.cat(all_lattices, dim=0)
num_atoms = torch.cat(all_num_atoms, dim=0)
num_known = torch.cat(all_num_known, dim=0)

---
## 4.5 Visualizing the diffusion trajectory

The diffusion model generates structures by denoising from pure noise. The `traj`
object captures **every intermediate step** — we can visualize the full trajectory
to see atoms crystallize from a random cloud into an ordered structure.

Below we render key frames using PyVista, creating a filmstrip that shows
the noise → crystal transition. The constrained kagome atoms are highlighted.

In [ ]:
import sys, os
from PIL import Image
from IPython.display import display as ipy_display

PROJECT_DIR = os.environ.get('PROJECT_ROOT', '/content/APS_demo_SCIGEN')
if os.path.join(PROJECT_DIR, 'scripts') not in sys.path:
    sys.path.insert(0, os.path.join(PROJECT_DIR, 'scripts'))
if os.path.exists(os.path.join(PROJECT_DIR, '.git')):
    os.system(f'cd {PROJECT_DIR} && git pull -q 2>/dev/null || true')

# --- Extract trajectory for the first structure ---
traj_coords = traj['all_frac_coords'].detach().cpu()   # (T+1, total_atoms, 3)
traj_lattices = traj['all_lattices'].detach().cpu()     # (T+1, n_struct, 3, 3)
traj_types = traj['atom_types'].detach().cpu()          # (T+1, total_atoms)
traj_num_atoms = traj['num_atoms'].detach().cpu()
traj_num_known = traj['num_known'].detach().cpu().flatten()

n_steps = traj_coords.shape[0]
na_first = int(traj_num_atoms[0].item())
nk_first = int(traj_num_known[0].item()) if traj_num_known.numel() > 0 else 0

# Select ~8 evenly spaced frames from noise to final structure
n_frames = min(8, n_steps)
frame_indices = np.linspace(0, n_steps - 1, n_frames, dtype=int)

print(f'Trajectory: {n_steps} diffusion steps, showing {n_frames} frames')
print(f'Structure 0: {na_first} atoms ({nk_first} constrained)')

try:
    from crystal_viz import plot_crystal

    # Build pymatgen structures at each trajectory frame
    from pymatgen.core.structure import Structure
    from pymatgen.core.lattice import Lattice as PmgLattice
    from scigen.common.data_utils import chemical_symbols

    frame_images = []
    for fi, step_idx in enumerate(frame_indices):
        # Extract coords and lattice for structure 0
        frac_i = traj_coords[step_idx, :na_first].numpy() % 1.0  # wrap to [0, 1)
        lat_mat = traj_lattices[step_idx, 0].numpy()  # 3x3 matrix
        types_i = traj_types[step_idx, :na_first].int().tolist()

        # Compute lengths and angles from lattice matrix
        lengths_i = np.sqrt(np.sum(lat_mat ** 2, axis=-1)).tolist()
        angles_i = []
        for d in range(3):
            j, k = (d + 1) % 3, (d + 2) % 3
            cos_a = np.clip(np.dot(lat_mat[j], lat_mat[k]) /
                            (lengths_i[j] * lengths_i[k] + 1e-10), -1, 1)
            angles_i.append(np.degrees(np.arccos(cos_a)))

        species = [chemical_symbols[t] for t in types_i]

        try:
            lattice = PmgLattice.from_parameters(*lengths_i, *angles_i)
            struct_frame = Structure(lattice, species, frac_i,
                                    coords_are_cartesian=False)
            highlight = list(range(nk_first)) if nk_first > 0 else None

            t_frac = step_idx / max(n_steps - 1, 1)
            title = f'Step {step_idx}/{n_steps-1} (t={1-t_frac:.2f})'

            img = plot_crystal(struct_frame, title=title,
                               atom_scale=0.25, elevation=70, azimuth=30,
                               highlight_indices=highlight,
                               window_size=(400, 350), display=False)
            frame_images.append(img)
        except Exception as e:
            print(f'  Frame {fi} (step {step_idx}): failed — {e}')

    # Stitch frames into a filmstrip
    if frame_images:
        ncols = min(4, len(frame_images))
        nrows = (len(frame_images) + ncols - 1) // ncols
        w, h = frame_images[0].size
        filmstrip = Image.new('RGB', (ncols * w, nrows * h), 'white')
        for i, img in enumerate(frame_images):
            r, c = divmod(i, ncols)
            filmstrip.paste(img.resize((w, h), Image.LANCZOS), (c * w, r * h))

        ipy_display(filmstrip)
        print(f'\nDiffusion trajectory: noise (top-left) → crystal (bottom-right)')
        print(f'Red-highlighted atoms = constrained {SC_TYPE} sites ({ELEMENT})')
    else:
        print('No frames rendered.')

except ImportError as e:
    print(f'PyVista not available ({e}) — using matplotlib fallback.')

    # Matplotlib fallback: show key frames as 3D scatter plots
    from scigen.common.data_utils import chemical_symbols
    n_show = min(4, len(frame_indices))
    fig, axes_traj = plt.subplots(1, n_show, figsize=(5 * n_show, 5),
                                   subplot_kw={'projection': '3d'})
    if not hasattr(axes_traj, '__len__'):
        axes_traj = [axes_traj]

    for fi, (ax_t, step_idx) in enumerate(zip(axes_traj, frame_indices[:n_show])):
        frac_i = traj_coords[step_idx, :na_first].numpy() % 1.0
        lat_mat = traj_lattices[step_idx, 0].numpy()
        cart = frac_i @ lat_mat

        ax_t.scatter(cart[:, 0], cart[:, 1], cart[:, 2],
                     s=100, c='steelblue', alpha=0.7, edgecolors='k', linewidths=0.5)
        if nk_first > 0:
            ax_t.scatter(cart[:nk_first, 0], cart[:nk_first, 1], cart[:nk_first, 2],
                         s=250, facecolors='none', edgecolors='red', linewidths=2)

        t_frac = step_idx / max(n_steps - 1, 1)
        ax_t.set_title(f'Step {step_idx} (t={1-t_frac:.2f})', fontsize=10)
        ax_t.set_xlabel('x'); ax_t.set_ylabel('y'); ax_t.set_zlabel('z')

    plt.suptitle(f'Diffusion trajectory ({SC_TYPE}): noise → crystal', fontsize=13, y=1.02)
    plt.tight_layout()
    plt.show()

except Exception as e:
    print(f'Trajectory visualization failed: {e}')

In [ ]:
# --- Animated GIF of the diffusion trajectory ---
# Render more frames and assemble into an animated GIF shown inline

try:
    from crystal_viz import plot_crystal
    import io

    n_anim_frames = min(20, n_steps)
    anim_indices = np.linspace(0, n_steps - 1, n_anim_frames, dtype=int)
    anim_images = []

    print(f'Rendering {n_anim_frames}-frame animation...')
    for step_idx in anim_indices:
        frac_i = traj_coords[step_idx, :na_first].numpy() % 1.0
        lat_mat = traj_lattices[step_idx, 0].numpy()
        types_i = traj_types[step_idx, :na_first].int().tolist()

        lengths_i = np.sqrt(np.sum(lat_mat ** 2, axis=-1)).tolist()
        angles_i = []
        for d in range(3):
            j, k = (d + 1) % 3, (d + 2) % 3
            cos_a = np.clip(np.dot(lat_mat[j], lat_mat[k]) /
                            (lengths_i[j] * lengths_i[k]), -1, 1)
            angles_i.append(np.degrees(np.arccos(cos_a)))

        species = [chemical_symbols[t] for t in types_i]
        try:
            lattice = PmgLattice.from_parameters(*lengths_i, *angles_i)
            struct_f = Structure(lattice, species, frac_i, coords_are_cartesian=False)
            highlight = list(range(nk_first)) if nk_first > 0 else None
            t_frac = step_idx / max(n_steps - 1, 1)

            img = plot_crystal(struct_f,
                               title=f't = {1-t_frac:.2f}',
                               atom_scale=0.25, elevation=70, azimuth=30,
                               highlight_indices=highlight,
                               window_size=(500, 450), display=False)
            anim_images.append(img)
        except Exception:
            pass

    if len(anim_images) > 2:
        # Save as animated GIF
        gif_buf = io.BytesIO()
        anim_images[0].save(gif_buf, format='GIF', save_all=True,
                            append_images=anim_images[1:],
                            duration=300, loop=0)
        gif_buf.seek(0)

        from IPython.display import Image as IPyImage, display
        display(IPyImage(data=gif_buf.read(), format='gif'))
        print(f'Animation: {len(anim_images)} frames, t = 1.0 (noise) → 0.0 (crystal)')
    else:
        print('Too few frames for animation.')

except ImportError:
    print('PyVista not available — animation skipped.')
except Exception as e:
    print(f'Animation failed: {e}')

---
## 4.6 Inspect the generated structures

Let's convert the raw tensor outputs to pymatgen `Structure` objects and examine
what the model produced. For each structure we will see:
- **Composition** — which elements did the model choose to complement the constrained sites?
- **Lattice parameters** — are the lengths and angles physically reasonable?
- **Known vs. total atoms** — how many atoms are from the constraint vs. model-generated?

The table below lists all generated structures with their key properties.

In [ ]:
def lattices_to_params(lattices):
    """Convert 3x3 lattice matrices to (lengths, angles) parameterization."""
    lengths = torch.sqrt(torch.sum(lattices ** 2, dim=-1))
    angles = torch.zeros_like(lengths)
    for i in range(3):
        j, k = (i + 1) % 3, (i + 2) % 3
        cos_angle = torch.clamp(
            torch.sum(lattices[..., j, :] * lattices[..., k, :], dim=-1)
            / (lengths[..., j] * lengths[..., k]),
            -1.0, 1.0,
        )
        angles[..., i] = torch.arccos(cos_angle) * 180.0 / np.pi
    return lengths, angles


lengths, angles = lattices_to_params(lattices)
num_known_flat = num_known.flatten()

from scigen.common.data_utils import chemical_symbols
from collections import Counter

print(f'Generated {num_atoms.shape[0]} structures\n')
print(f'{"#":>3} {"Atoms":>5} {"Known":>5} {"Composition":<25} {"a":>7} {"b":>7} {"c":>7}  {"α":>6} {"β":>6} {"γ":>6}')
print('-' * 95)

start_idx = 0
for i in range(num_atoms.shape[0]):
    na = int(num_atoms[i])
    nk = int(num_known_flat[i]) if i < len(num_known_flat) else 0
    types_i = atom_types[start_idx:start_idx + na].int().tolist()
    symbols = [chemical_symbols[t] for t in types_i]
    comp = Counter(symbols)
    comp_str = ' '.join(f'{el}{n}' for el, n in sorted(comp.items()))
    l = lengths[i].tolist()
    a = angles[i].tolist()
    print(f'{i:3d} {na:5d} {nk:5d} {comp_str:<25} {l[0]:7.2f} {l[1]:7.2f} {l[2]:7.2f}  {a[0]:6.1f} {a[1]:6.1f} {a[2]:6.1f}')
    start_idx += na

In [ ]:
# Convert to pymatgen Structure objects
from pymatgen.core.structure import Structure
from pymatgen.core.lattice import Lattice

structures = []
start_idx = 0

for i in range(num_atoms.shape[0]):
    na = int(num_atoms[i])
    types_i = atom_types[start_idx:start_idx + na].int().tolist()
    coords_i = frac_coords[start_idx:start_idx + na].numpy()
    lengths_i = lengths[i].tolist()
    angles_i = angles[i].tolist()

    species = [chemical_symbols[t] for t in types_i]
    lattice = Lattice.from_parameters(*lengths_i, *angles_i)

    try:
        structure = Structure(lattice, species, coords_i, coords_are_cartesian=False)
        structures.append(structure)
        print(f'Structure {i}: {structure.composition.reduced_formula}'
              f'  ({na} atoms, V={structure.volume:.1f} Å³)')
    except Exception as e:
        print(f'Structure {i}: construction failed - {e}')

    start_idx += na

print(f'\nSuccessfully constructed {len(structures)}/{num_atoms.shape[0]} structures')

### 4.6.1 Lattice parameter distributions

The plots below show the distribution of lattice lengths (a, b, c) and angles
(α, β, γ) for all generated structures. For constrained lattices like kagome,
you should see a ≈ b with γ ≈ 120° (hexagonal symmetry). Physically reasonable
lattice lengths are typically 2–15 Å, and angles cluster around 60°, 90°, or 120°.

In [ ]:
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

length_data = lengths.numpy()
angle_data = angles.numpy()
n_samples = len(length_data)

if n_samples >= 20:
    # Enough data for histograms
    for j, label in enumerate(['a', 'b', 'c']):
        axes[0].hist(length_data[:, j], bins=15, alpha=0.6, label=label, edgecolor='white')
    for j, label in enumerate(['\u03b1', '\u03b2', '\u03b3']):
        axes[1].hist(angle_data[:, j], bins=15, alpha=0.6, label=label, edgecolor='white')
    axes[0].set_ylabel('Count')
    axes[1].set_ylabel('Count')
else:
    # Few samples: use strip plot (jittered scatter) so every point is visible
    colors = plt.cm.tab10.colors
    for j, label in enumerate(['a', 'b', 'c']):
        y = np.random.default_rng(j).uniform(-0.3, 0.3, n_samples) + j
        axes[0].scatter(length_data[:, j], y, s=120, alpha=0.8,
                        color=colors[j], edgecolors='k', linewidths=0.5,
                        label=label, zorder=3)
    axes[0].set_yticks(range(3))
    axes[0].set_yticklabels(['a', 'b', 'c'])
    for j, label in enumerate(['\u03b1', '\u03b2', '\u03b3']):
        y = np.random.default_rng(j).uniform(-0.3, 0.3, n_samples) + j
        axes[1].scatter(angle_data[:, j], y, s=120, alpha=0.8,
                        color=colors[j], edgecolors='k', linewidths=0.5,
                        label=label, zorder=3)
    axes[1].set_yticks(range(3))
    axes[1].set_yticklabels(['\u03b1', '\u03b2', '\u03b3'])
    axes[0].grid(axis='x', alpha=0.3)
    axes[1].grid(axis='x', alpha=0.3)

axes[0].set_xlabel('Length (\u00c5)')
axes[0].set_title(f'Lattice lengths (n={n_samples})')
axes[0].legend()

axes[1].set_xlabel('Angle (\u00b0)')
axes[1].set_title(f'Lattice angles (n={n_samples})')
axes[1].legend()

plt.tight_layout()
plt.show()

In [ ]:
try:
    from crystal_viz import plot_crystal, plot_crystal_grid, plot_crystal_interactive

    # Interactive viewer for the first generated structure
    if structures:
        viewer = plot_crystal_interactive(
            structures[0],
            title=f'{structures[0].composition.reduced_formula} ({SC_TYPE})',
            atom_scale_default=0.3, elevation_default=70,
        )
        display(viewer)

    # Grid view of all generated structures
    if len(structures) > 1:
        titles = [f'#{i}: {s.composition.reduced_formula} ({SC_TYPE})'
                  for i, s in enumerate(structures)]
        grid_img = plot_crystal_grid(structures, titles=titles,
                                     ncols=min(4, len(structures)),
                                     atom_scale=0.3, elevation=70)
except ImportError as e:
    print(f'PyVista not available ({e}) — skipping 3D rendering.')
except Exception as e:
    print(f'3D rendering failed: {e}')

---
## 4.7 Space group analysis

Let's determine the symmetry of our generated structures. The space group tells us
about the crystal's symmetry operations and is a key descriptor for comparing against
known materials.

In [ ]:
from pymatgen.symmetry.analyzer import SpacegroupAnalyzer

print(f'{"#":>3} {"Formula":<20} {"SG Symbol":<12} {"SG #":>5} {"Crystal System":<16} {"Point Group":<12}')
print('-' * 75)

sg_numbers = []
crystal_systems = []

for i, s in enumerate(structures):
    try:
        sga = SpacegroupAnalyzer(s, symprec=0.1)
        sg_sym = sga.get_space_group_symbol()
        sg_num = sga.get_space_group_number()
        csys = sga.get_crystal_system()
        pg = sga.get_point_group_symbol()
        sg_numbers.append(sg_num)
        crystal_systems.append(csys)
        print(f'{i:3d} {s.composition.reduced_formula:<20} {sg_sym:<12} {sg_num:5d} {csys:<16} {pg:<12}')
    except Exception as e:
        sg_numbers.append(None)
        crystal_systems.append(None)
        print(f'{i:3d} {s.composition.reduced_formula:<20} {"Error":<12} {"":>5} {str(e)[:30]}')

In [ ]:
import matplotlib.pyplot as plt

valid_sg = [sg for sg in sg_numbers if sg is not None]
valid_cs = [cs for cs in crystal_systems if cs is not None]

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Space group number histogram
if valid_sg:
    axes[0].hist(valid_sg, bins=range(1, 232), color='steelblue', edgecolor='white')
    axes[0].set_xlabel('Space group number', fontsize=11)
    axes[0].set_ylabel('Count', fontsize=11)
    axes[0].set_title('Space groups of generated structures', fontsize=12)
    axes[0].set_xlim(0, 231)
else:
    axes[0].text(0.5, 0.5, 'No valid structures', ha='center', va='center', transform=axes[0].transAxes)

# Crystal system pie chart
if valid_cs:
    from collections import Counter
    cs_counts = Counter(valid_cs)
    cs_colors = {'triclinic': '#e6194b', 'monoclinic': '#f58231', 'orthorhombic': '#ffe119',
                 'tetragonal': '#3cb44b', 'trigonal': '#42d4f4', 'hexagonal': '#4363d8', 'cubic': '#911eb4'}
    labels = list(cs_counts.keys())
    sizes = list(cs_counts.values())
    colors = [cs_colors.get(l, '#999999') for l in labels]
    axes[1].pie(sizes, labels=labels, colors=colors, autopct='%1.0f%%', startangle=90)
    axes[1].set_title('Crystal system distribution', fontsize=12)
else:
    axes[1].text(0.5, 0.5, 'No valid structures', ha='center', va='center', transform=axes[1].transAxes)

plt.tight_layout()
plt.show()

---
## 4.8 Simulated X-ray diffraction

Simulated **powder XRD patterns** are a fingerprint of each crystal structure.
Comparing the XRD pattern of a generated structure against experimental data is
a key validation step. Here we use pymatgen's `XRDCalculator` with Cu Ka radiation.

In [ ]:
from pymatgen.analysis.diffraction.xrd import XRDCalculator

xrd_calc = XRDCalculator(wavelength='CuKa')

n_xrd = min(4, len(structures))
fig, axes = plt.subplots(n_xrd, 1, figsize=(10, 3 * n_xrd), sharex=True)
if n_xrd == 1:
    axes = [axes]

colors = ['steelblue', 'coral', '#2ca02c', '#9467bd']

for i in range(n_xrd):
    try:
        pattern = xrd_calc.get_pattern(structures[i])
        axes[i].stem(pattern.x, pattern.y, linefmt=colors[i % len(colors)],
                     markerfmt=' ', basefmt=' ')
        axes[i].set_ylabel('Intensity (%)', fontsize=10)
        formula = structures[i].composition.reduced_formula
        axes[i].set_title(f'#{i}: {formula}', fontsize=11, loc='left')
        axes[i].set_xlim(10, 90)
        axes[i].set_ylim(0, 110)
        # Label strongest peaks
        for x, y, hkl in zip(pattern.x, pattern.y, pattern.hkls):
            if y > 30:
                label = ''.join(str(v) for v in hkl[0]['hkl'])
                axes[i].annotate(f'({label})', xy=(x, y), xytext=(x, y + 5),
                                 fontsize=6, ha='center', color='gray')
    except Exception as e:
        axes[i].text(0.5, 0.5, f'XRD failed: {e}', ha='center', va='center',
                     transform=axes[i].transAxes, fontsize=9)

axes[-1].set_xlabel('2θ (degrees)', fontsize=11)
plt.tight_layout()
plt.show()

---
## 4.9 Comparing generated vs published SCIGEN structures

The SCIGEN paper reports generating **10 million candidate structures**, screened down
to **24,743 DFT-validated materials**. We can download these DFT-relaxed structures
and compare their property distributions against:
- The **MP-20 training data** (what the model learned from)
- Our **freshly generated structures** (from this notebook)

This three-way comparison reveals how well SCIGEN covers materials space.

In [ ]:
# takes long (~1-3 min) — downloads 28 MB of SCIGEN DFT-relaxed structures

# === Download SCIGEN DFT-relaxed structures (24,743 materials) ===
import os, zipfile
from urllib.request import urlretrieve

# Use Figshare API endpoint (the ndownloader URL is blocked by AWS WAF)
SCIGEN_DFT_URL = 'https://api.figshare.com/v2/file/download/57245942'
SCIGEN_DFT_DIR = os.path.join(PROJECT_DIR, 'data', 'scigen_generated', '03_scigen_materials_relaxed')

if not os.path.exists(SCIGEN_DFT_DIR) or len(os.listdir(SCIGEN_DFT_DIR)) < 100:
    zip_path = os.path.join(PROJECT_DIR, 'data', 'scigen_generated', 'scigen_dft.zip')
    os.makedirs(os.path.dirname(zip_path), exist_ok=True)

    if not os.path.exists(zip_path) or os.path.getsize(zip_path) < 1000:
        print('Downloading SCIGEN DFT-relaxed structures (~28 MB)...')
        import time
        for attempt in range(5):
            try:
                urlretrieve(SCIGEN_DFT_URL, zip_path)
                if os.path.getsize(zip_path) > 1000:
                    print(f'Downloaded: {os.path.getsize(zip_path)/1e6:.1f} MB')
                    break
                time.sleep(3)
            except Exception as e:
                print(f'Attempt {attempt+1}: {e}')
                time.sleep(3)

    if os.path.exists(zip_path) and os.path.getsize(zip_path) > 1000:
        print('Extracting...')
        with zipfile.ZipFile(zip_path, 'r') as zf:
            zf.extractall(os.path.dirname(SCIGEN_DFT_DIR))
        print(f'Extracted to {SCIGEN_DFT_DIR}')
    else:
        print('Download failed — Figshare may be slow. Try running this cell again.')

if os.path.exists(SCIGEN_DFT_DIR):
    n_scigen = len([d for d in os.listdir(SCIGEN_DFT_DIR) if os.path.isdir(os.path.join(SCIGEN_DFT_DIR, d))])
    print(f'SCIGEN DFT structures available: {n_scigen}')

    # Count by constraint type
    from collections import Counter
    sc_types = [d.split('_')[0] for d in os.listdir(SCIGEN_DFT_DIR)
                if os.path.isdir(os.path.join(SCIGEN_DFT_DIR, d))]
    sc_counts = Counter(sc_types)
    print('\nBy structural constraint:')
    for sc, count in sc_counts.most_common():
        print(f'  {sc:<6s} {count:>5d}')

In [ ]:
# takes long (~5-10 min) — parses 500 SCIGEN structures + 500 training structures

# === Parse SCIGEN DFT structures (sample for speed) ===
from pymatgen.io.vasp import Poscar
from pymatgen.core import Structure
import numpy as np
import random

random.seed(42)
all_dirs = sorted([d for d in os.listdir(SCIGEN_DFT_DIR)
                    if os.path.isdir(os.path.join(SCIGEN_DFT_DIR, d))])

# Sample up to 500 for speed (parsing 24K CONTCARs would be slow)
sample_dirs = random.sample(all_dirs, min(500, len(all_dirs)))

scigen_structures = []
scigen_nn_dists = []
scigen_elements = []
scigen_sc_types = []

print(f'Parsing {len(sample_dirs)} SCIGEN DFT structures...')
for d in sample_dirs:
    contcar = os.path.join(SCIGEN_DFT_DIR, d, 'CONTCAR')
    if not os.path.exists(contcar):
        continue
    try:
        s = Poscar.from_file(contcar).structure
        scigen_structures.append(s)
        scigen_sc_types.append(d.split('_')[0])
        elems = [str(sp.specie) for sp in s]
        scigen_elements.extend(elems)
        for site in s:
            neighbors = s.get_neighbors(site, r=4.0)
            if neighbors:
                scigen_nn_dists.append(min(n.nn_distance for n in neighbors))
    except Exception:
        pass

print(f'Parsed {len(scigen_structures)} SCIGEN structures')

# Also load MP-20 training data for comparison
import pandas as pd
train_df = pd.read_csv(os.path.join(PROJECT_DIR, 'data', 'mp_20', 'train.csv'))

n_train_sample = min(500, len(train_df))
train_structures_cmp = []
train_nn_dists_cmp = []
train_elements_cmp = []

print(f'Parsing {n_train_sample} MP-20 training structures...')
for i in range(n_train_sample):
    try:
        s = Structure.from_str(train_df.iloc[i]['cif'], fmt='cif')
        train_structures_cmp.append(s)
        elems = [str(sp.specie) for sp in s]
        train_elements_cmp.extend(elems)
        for site in s:
            neighbors = s.get_neighbors(site, r=4.0)
            if neighbors:
                train_nn_dists_cmp.append(min(n.nn_distance for n in neighbors))
    except Exception:
        pass

print(f'Parsed {len(train_structures_cmp)} training structures')

In [ ]:
# === Distribution comparison: MP-20 vs SCIGEN DFT vs freshly generated ===
from collections import Counter

fig, axes = plt.subplots(2, 3, figsize=(16, 10))

# 1. Atoms per cell
ax = axes[0, 0]
train_na = [s.num_sites for s in train_structures_cmp]
scigen_na = [s.num_sites for s in scigen_structures]
gen_na = num_atoms.numpy() if 'num_atoms' in dir() else []
ax.hist(train_na, bins=range(1, 22), alpha=0.5, color='steelblue', edgecolor='white',
        label='MP-20 training', density=True)
ax.hist(scigen_na, bins=range(1, 22), alpha=0.5, color='seagreen', edgecolor='white',
        label='SCIGEN DFT (24K)', density=True)
if len(gen_na) > 0:
    ax.hist(gen_na, bins=range(1, 22), alpha=0.5, color='coral', edgecolor='white',
            label='This notebook', density=True)
ax.set_xlabel('Atoms per unit cell'); ax.set_ylabel('Density')
ax.set_title('Unit cell size'); ax.legend(fontsize=8)

# 2. Lattice parameter a
ax = axes[0, 1]
train_a = [s.lattice.a for s in train_structures_cmp]
scigen_a = [s.lattice.a for s in scigen_structures]
ax.hist(train_a, bins=25, alpha=0.5, color='steelblue', edgecolor='white',
        label='MP-20', density=True, range=(2, 15))
ax.hist(scigen_a, bins=25, alpha=0.5, color='seagreen', edgecolor='white',
        label='SCIGEN DFT', density=True, range=(2, 15))
ax.set_xlabel('Lattice a (Ang)'); ax.set_ylabel('Density')
ax.set_title('Lattice parameter a'); ax.legend(fontsize=8)

# 3. Lattice parameter c
ax = axes[0, 2]
train_c = [s.lattice.c for s in train_structures_cmp]
scigen_c = [s.lattice.c for s in scigen_structures]
ax.hist(train_c, bins=25, alpha=0.5, color='steelblue', edgecolor='white',
        label='MP-20', density=True, range=(2, 20))
ax.hist(scigen_c, bins=25, alpha=0.5, color='seagreen', edgecolor='white',
        label='SCIGEN DFT', density=True, range=(2, 20))
ax.set_xlabel('Lattice c (Ang)'); ax.set_ylabel('Density')
ax.set_title('Lattice parameter c'); ax.legend(fontsize=8)

# 4. Bond lengths
ax = axes[1, 0]
ax.hist(train_nn_dists_cmp, bins=30, alpha=0.5, color='steelblue', edgecolor='white',
        label='MP-20', density=True, range=(0.5, 5.0))
ax.hist(scigen_nn_dists, bins=30, alpha=0.5, color='seagreen', edgecolor='white',
        label='SCIGEN DFT', density=True, range=(0.5, 5.0))
ax.axvspan(1.5, 3.5, alpha=0.1, color='green')
ax.set_xlabel('NN distance (Ang)'); ax.set_ylabel('Density')
ax.set_title('Bond length distribution'); ax.legend(fontsize=8)

# 5. Element frequency
ax = axes[1, 1]
train_ec = Counter(train_elements_cmp)
scigen_ec = Counter(scigen_elements)
top_el = [el for el, _ in train_ec.most_common(15)]
x = np.arange(len(top_el))
t_freq = np.array([train_ec.get(e, 0) for e in top_el], dtype=float)
s_freq = np.array([scigen_ec.get(e, 0) for e in top_el], dtype=float)
t_freq /= max(t_freq.sum(), 1); s_freq /= max(s_freq.sum(), 1)
ax.bar(x - 0.2, t_freq, 0.4, color='steelblue', label='MP-20', alpha=0.7)
ax.bar(x + 0.2, s_freq, 0.4, color='seagreen', label='SCIGEN DFT', alpha=0.7)
ax.set_xticks(x); ax.set_xticklabels(top_el, rotation=45, ha='right', fontsize=8)
ax.set_ylabel('Frequency'); ax.set_title('Element distribution (top 15)')
ax.legend(fontsize=8)

# 6. Constraint type breakdown (SCIGEN only)
ax = axes[1, 2]
sc_counts = Counter(scigen_sc_types)
sc_labels = [k for k, _ in sc_counts.most_common()]
sc_values = [v for _, v in sc_counts.most_common()]
colors_sc = plt.cm.Set3(np.linspace(0, 1, len(sc_labels)))
ax.barh(sc_labels, sc_values, color=colors_sc, edgecolor='white')
ax.set_xlabel('Count (in sample)')
ax.set_title(f'SCIGEN constraint types (n={len(scigen_structures)})')
ax.invert_yaxis()

plt.suptitle('MP-20 Training (blue) vs SCIGEN DFT-validated (green)', fontsize=14, y=1.02)
plt.tight_layout()
plt.show()

print(f'MP-20 training: {len(train_structures_cmp)} structures')
print(f'SCIGEN DFT:     {len(scigen_structures)} structures (sampled from 24,743)')
if scigen_nn_dists:
    frac = sum(1 for d in scigen_nn_dists if 1.0 < d < 4.0) / len(scigen_nn_dists)
    print(f'SCIGEN physical NN distances (1-4 Ang): {frac*100:.0f}%')

In [ ]:
# === t-SNE: composition space (MP-20 vs SCIGEN DFT) ===
from sklearn.manifold import TSNE

MAX_Z = 84

def composition_vector(structure):
    vec = np.zeros(MAX_Z)
    comp = structure.composition.fractional_composition
    for el in comp:
        z = el.Z
        if z < MAX_Z:
            vec[z] = comp[el]
    return vec

train_vecs = np.array([composition_vector(s) for s in train_structures_cmp])
scigen_vecs = np.array([composition_vector(s) for s in scigen_structures])

# Also include freshly generated structures if available
gen_vecs = np.array([composition_vector(s) for s in structures]) if structures else np.empty((0, MAX_Z))

all_vecs = np.vstack([v for v in [train_vecs, scigen_vecs, gen_vecs] if len(v) > 0])
n_train = len(train_vecs)
n_scigen = len(scigen_vecs)
n_gen = len(gen_vecs)

print(f't-SNE: {n_train} training + {n_scigen} SCIGEN DFT + {n_gen} generated = {len(all_vecs)} total')

perplexity = min(30, len(all_vecs) - 1)
tsne = TSNE(n_components=2, perplexity=perplexity, random_state=42, n_iter=1000)
emb = tsne.fit_transform(all_vecs)

fig, ax = plt.subplots(figsize=(9, 7))
ax.scatter(emb[:n_train, 0], emb[:n_train, 1],
           s=15, c='steelblue', alpha=0.3, label=f'MP-20 training (n={n_train})', zorder=2)
ax.scatter(emb[n_train:n_train+n_scigen, 0], emb[n_train:n_train+n_scigen, 1],
           s=25, c='seagreen', alpha=0.5, label=f'SCIGEN DFT (n={n_scigen})', zorder=3,
           edgecolors='darkgreen', linewidths=0.3)
if n_gen > 0:
    ax.scatter(emb[n_train+n_scigen:, 0], emb[n_train+n_scigen:, 1],
               s=80, c='coral', alpha=0.9, marker='*', label=f'This notebook (n={n_gen})',
               zorder=4, edgecolors='black', linewidths=0.5)

ax.set_xlabel('t-SNE 1', fontsize=12); ax.set_ylabel('t-SNE 2', fontsize=12)
ax.set_title('Composition space: MP-20 vs SCIGEN DFT-validated structures', fontsize=13)
ax.legend(fontsize=10, markerscale=1.5)
ax.grid(True, alpha=0.15)
plt.tight_layout()
plt.show()

print('SCIGEN DFT structures (green) should overlap with training data (blue)')
print('while also exploring new regions of composition space.')

---
## 4.10 Export as CIF files

Save the generated structures as CIF files for further analysis
(e.g., DFT relaxation, MLIP evaluation from Notebook 05).

In [ ]:
output_dir = os.path.join(PROJECT_DIR, 'generated_cifs')
os.makedirs(output_dir, exist_ok=True)

for i, structure in enumerate(structures):
    formula = structure.composition.reduced_formula
    cif_path = os.path.join(output_dir, f'{SC_TYPE}_{formula}_{i:03d}.cif')
    structure.to(filename=cif_path, fmt='cif')
    print(f'Saved: {os.path.basename(cif_path)}')

print(f'\n{len(structures)} CIF files saved to {output_dir}')

In [ ]:
import matplotlib.patches as mpatches

# Pipeline flowchart
fig, ax = plt.subplots(figsize=(12, 2))
ax.axis('off')

steps = ['Choose\nconstraint', 'Generate\n(SCIGEN)', 'Pre-screen\n(bonds, comp.)', 
         'MLIP screen\n(CHGNet)', 'DFT\nvalidation', 'Experimental\nsynthesis']
colors = ['#4ECDC4', '#45B7D1', '#96CEB4', '#FFEAA7', '#DDA0DD', '#98D8C8']

for i, (step, color) in enumerate(zip(steps, colors)):
    x = i * 1.8
    rect = mpatches.FancyBboxPatch((x, 0.2), 1.4, 0.6, boxstyle='round,pad=0.1',
                                     facecolor=color, edgecolor='gray', linewidth=1.5)
    ax.add_patch(rect)
    ax.text(x + 0.7, 0.5, step, ha='center', va='center', fontsize=9, fontweight='bold')
    if i < len(steps) - 1:
        ax.annotate('', xy=(x + 1.55, 0.5), xytext=(x + 1.45, 0.5),
                     arrowprops=dict(arrowstyle='->', color='gray', lw=2))

ax.set_xlim(-0.3, len(steps) * 1.8)
ax.set_ylim(-0.1, 1.1)
ax.set_title('SCIGEN discovery pipeline', fontsize=13, pad=10)
plt.tight_layout()
plt.show()

In [ ]:
# Download CIF files as a zip archive (Colab only)
try:
    import shutil
    from google.colab import files
    zip_name = f'scigen_{SC_TYPE}_{ELEMENT}'
    zip_path = shutil.make_archive(
        os.path.join(PROJECT_DIR, zip_name), 'zip', output_dir
    )
    files.download(zip_path)
    print(f'Downloading {zip_name}.zip')
except ImportError:
    print('Not running in Colab — skipping download.')
except Exception as e:
    print(f'Download failed: {e}')
    print(f'CIF files are available at: {output_dir}')

---
## 4.11 (Optional) Quick evaluation with CHGNet

For a quick stability check, you can predict the formation energy of your generated
structures using CHGNet (covered in depth in Notebook 05). If CHGNet is installed,
the cell below will predict energy and forces for the first structure.

For a full evaluation — relaxation, phonon stability, convex hull analysis, and
screening across constraint types — proceed to **Notebook 05**.

---
## 4.12 Try it yourself!

Go back to **Section 4.4** and change the parameters:

- **Different lattice:** Change `SC_TYPE` to `'hon'` (honeycomb), `'tri'` (triangular), `'sqr'` (square), etc.
- **Different element:** Change `ELEMENT` to `'Fe'`, `'Co'`, `'Ni'`, `'Ti'`, etc.
- **More structures:** Increase `BATCH_SIZE` or `NUM_BATCHES`
- **Custom constraints:** See the [README](https://github.com/RyotaroOKabe/APS_demo_SCIGEN#create-your-own-structural-constraint) for defining your own lattice types

Then re-run from Section 4.5 onward.

### 4.12.1 Exercises

1. Generate with `SC_TYPE = 'hon'` (honeycomb) using Fe. How do the compositions and lattice parameters compare to kagome Mn?
2. Try `STEP_LR = 1e-5` instead of `5e-6`. Does the generation quality change?
3. Look at the bond length histogram. Are any structures clearly unphysical (NN distance < 1.0 Å)?

---
## References

- **SCIGEN:** Okabe et al., "Structural constraint integration in a generative model for the discovery of quantum materials," *Nature Materials* (2025). [DOI](https://doi.org/10.1038/s41563-025-02355-y) | [GitHub](https://github.com/RyotaroOKabe/SCIGEN)
- **CHGNet:** Deng et al., "CHGNet as a pretrained universal neural network potential for charge-informed atomistic modelling," *Nature Machine Intelligence* 5, 1031–1041 (2023). [GitHub](https://github.com/CederGroupHub/chgnet)
- **SMACT:** Davies et al., "SMACT: Semiconducting Materials by Analogy and Chemical Theory," *JOSS* 4, 1361 (2019). [GitHub](https://github.com/WMD-group/SMACT)
- **pymatgen:** Ong et al., "Python Materials Genomics (pymatgen)," *Comput. Mater. Sci.* 68, 314–319 (2013). [GitHub](https://github.com/materialsproject/pymatgen)
- **PyVista:** Sullivan & Kaszynski, "PyVista: 3D plotting and mesh analysis," *JOSS* 4, 1450 (2019). [GitHub](https://github.com/pyvista/pyvista)

---
## Summary

In this tutorial, we:
1. **Represented** crystal structures with pymatgen (Notebook 01)
2. **Learned** how diffusion models generate crystals (Notebook 02)
3. **Understood** DiffCSP's noise and denoising process (Notebook 03)
4. **Generated** new crystal structures with SCIGEN's structural constraints (this notebook)

Proceed to **Notebook 05: MLIP Evaluation** to evaluate the generated structures
with CHGNet (energies, forces, phonons, thermodynamic stability).

### The full SCIGEN pipeline

```
Choose constraint (kagome, honeycomb, ...)
    -> Generate candidates with SCIGEN
    -> Pre-screen (composition, bond distances)
    -> MLIP evaluation (CHGNet, GNN models)
    -> DFT validation of best candidates
    -> Experimental synthesis
```

For more details, see the [SCIGEN paper](https://doi.org/10.1038/s41563-025-02355-y) (Nature Materials, 2025).